# Hands-on-Übung: (S)ARIMA(X) mit dem BHKW-Datensatz Pfettscher

> **Forschungsfrage: Verbessern Saisonalität & exogene Informationen den Forecast des täglichen Hausverbrauchs gegenüber einem ARIMA-Modell?**

Vorgehen
1. BHKW-Daten von Pfettscher laden, vorverarbeiten und Trainings-/Test-Split erstellen.
2. Modelle trainieren
    1. ARIMA-Modell als Referenzmodell zu trainieren
    2. SARIMA-Modell mit Erweiterung um Saisonalität trainieren.
    3. SARIMAX-Modell mit exogenen Variablen trainieren.
4. Drei Modelle vergleichen.
5. Modellannahme der stationären Residuen prüfen.
6. Forschungsfrage beantworten.

In [ ]:
# 0. Imports und Konfiguration
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Lokaler Datenordner
# Aufgabe: Passe den Pfad zu dem lokalen BHKW-Datenordner an
BHKW_BASE_DIR = Path(r"root\BHKWs_Daten\Pfettscher")

## 1. BHKW-Daten von Pfettscher laden, vorverarbeiten und Trainings-/Test-Split erstellen

In [ ]:
# Daten laden: alle CSV-Dateien im Ordner Pfettscher automatisch einlesen
def read_bhkw_csv(path: Path) -> pd.DataFrame:
    """Liest eine BHKW-CSV robust ein: Semikolon, Dezimalkomma, flexible Zeitspalte."""
    try:
        df = pd.read_csv(path, sep=";", decimal=",", encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(path, sep=";", decimal=",", encoding="latin1", low_memory=False)

    df.columns = [" ".join(str(c).split()) for c in df.columns]
    time_candidates = ["Zeitstempel", "Timestamp", "Datum", "Date", "datetime", "DateTime", "Zeit"]
    time_col = next((c for c in time_candidates if c in df.columns), None)
    if time_col is None:
        raise ValueError(f"Keine Zeitspalte in {path.name} gefunden. Spalten: {list(df.columns)}")
    df[time_col] = pd.to_datetime(df[time_col], dayfirst=True, errors="coerce")
    df = df.dropna(subset=[time_col]).set_index(time_col).sort_index()

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = (df[col].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

csv_files = sorted(BHKW_BASE_DIR.glob("*.csv"))

if not BHKW_BASE_DIR.exists():
    raise FileNotFoundError(f"Ordner nicht gefunden: {BHKW_BASE_DIR}")
if not csv_files:
    raise FileNotFoundError(f"Keine CSV-Dateien im Ordner gefunden: {BHKW_BASE_DIR}")

raw = pd.concat([read_bhkw_csv(f) for f in csv_files], axis=0).sort_index()
raw = raw[~raw.index.duplicated(keep="last")]

print(f"Geladene Anlage: {BHKW_BASE_DIR}")
print(f"Geladene Dateien: {[f.name for f in csv_files]}")
print(f"Zeitraum: {raw.index.min()} bis {raw.index.max()}")
print(f"Messpunkte: {len(raw):,}")
display(raw.head())

In [ ]:
# Target und exogene Variable
TARGET_COL = "Hausverbrauch [W]"
PRODUCTION_COL = "Summe Produktion [W]"

In [ ]:
# Zielvariable und exogene Variablen vorbereiten
# Stündliche W-Werte werden zu Tagesenergie in kWh/Tag aggregiert.
# Vorteil: schneller, stabiler und für die Übung besser interpretierbar.

required_cols = [TARGET_COL, PRODUCTION_COL]
missing = [c for c in required_cols if c not in raw.columns]
if missing:
    raise ValueError(f"Fehlende Pflichtspalten: {missing}. Verfügbare Spalten: {list(raw.columns)}")

def power_to_daily_kwh(series: pd.Series) -> pd.Series:
    return series.resample("D").sum(min_count=18) / 1000

model_df = pd.DataFrame(index=pd.date_range(raw.index.min().date(), raw.index.max().date(), freq="D"))
model_df["y_hausverbrauch_kwh"] = power_to_daily_kwh(raw[TARGET_COL])
model_df["produktion_kwh"] = power_to_daily_kwh(raw[PRODUCTION_COL])

# Fehlende Werte der Zielvariable werden nicht künstlich aufgefüllt, 
# da dies zu unrealistischen Beobachtungen führen und die Bewertung der Prognosegüte verfälschen würde. 
# Stattdessen werden unvollständige Tage aus dem Datensatz entfernt.
model_df = model_df.dropna(subset=["y_hausverbrauch_kwh", "produktion_kwh"])

print(model_df.info())
display(model_df.head())

In [ ]:
# Train/Test-Split und Metriken

TEST_DAYS = 90
train = model_df.iloc[:-TEST_DAYS].copy()
test = model_df.iloc[-TEST_DAYS:].copy()

y_train = train["y_hausverbrauch_kwh"]
y_test = test["y_hausverbrauch_kwh"]

print("Train:", train.index.min().date(), "bis", train.index.max().date(), "|", len(train), "Tage")
print("Test: ", test.index.min().date(), "bis", test.index.max().date(), "|", len(test), "Tage")

## 2. Modelle trainieren

In [ ]:
# Variablen, um die Ergebnisse der Modelle zu speichern und zu vergleichen
results = []
forecasts = pd.DataFrame(index=test.index)
forecasts["Ist"] = y_test

### 2.1. ARIMA als Referenzmodell trainieren

ARIMA modelliert Trend und Autokorrelation, aber keine explizite Saisonalität.

In [ ]:
# Aufgabe:
# Schreibe die ARIMA-Methode mit den gegeben Parametern y_train & ARIMA_ORDER.

ARIMA_ORDER = (1, 1, 1)
arima_model = ARIMA(_______)
arima_result = arima_model.fit()

# Forecast inklusive Konfidenzintervall
arima_forecast_result = arima_result.get_forecast(steps=len(y_test))
arima_forecast = arima_forecast_result.predicted_mean
arima_conf_int = arima_forecast_result.conf_int(alpha=0.05)
arima_forecast.index = y_test.index
arima_conf_int.index = y_test.index
forecasts["ARIMA"] = arima_forecast

print("ARIMA trainiert.")

<details>
  <summary>Lösung anzeigen</summary>
  
```python
arima_model = ARIMA(y_train, order=ARIMA_ORDER) 
```

</details>

#### Plot der ARIMA-Prognose mit Konfidenzinterval

In [ ]:
# Plot
plot_days = 90
plt.figure(figsize=(14, 5))
plt.plot(y_train.index[-plot_days:], y_train[-plot_days:], label="Train", color="blue")
plt.plot(y_test.index, y_test, label="Test / tatsächlicher Strombedarf", color="black")
plt.plot(arima_forecast.index, arima_forecast, label="ARIMA Forecast", color="red")
plt.fill_between(arima_conf_int.index, arima_conf_int.iloc[:, 0], arima_conf_int.iloc[:, 1], color="red", alpha=0.2, label="95%-Konfidenzintervall")
plt.title("ARIMA Forecast mit 95%-Konfidenzintervall")
plt.xlabel("Zeit")
plt.ylabel("Strombedarf")
plt.legend()
plt.grid(True, alpha=0.8)
plt.tight_layout()
plt.show()

Aufgaben zur Reflexion

1. Wo macht ARIMA systematische Fehler?
2. Welche Muster erkennt das Modell offenbar nicht?
3. Welche Erweiterung könnte helfen?

### 2.2 SARIMA trainieren: nutzt nur die Vergangenheit der Zielvariable

In [ ]:
# Aufgabe:
# Schreibe die SARIMA-Methode mit den gegeben Parametern y_train, SARIMA_ORDER & SEASONAL_ORDER.

SEASONAL_PERIOD = 7    
SARIMA_ORDER = (1, 0, 1)                         
SEASONAL_ORDER = (1, 1, 1, SEASONAL_PERIOD)   

sarima_model = SARIMAX(_______)
sarima_result = sarima_model.fit(disp=False)

# Forecast erzeugen
sarima_forecast_result = sarima_result.get_forecast(steps=len(y_test))
sarima_forecast = sarima_forecast_result.predicted_mean
sarima_conf_int = sarima_forecast_result.conf_int(alpha=0.05)
sarima_forecast.index = y_test.index
sarima_conf_int.index = y_test.index
forecasts["SARIMA"] = sarima_forecast

print("SARIMA trainiert.")

<details>
  <summary>Lösung anzeigen</summary>

```python
sarima_model = SARIMAX(
    y_train,
    order=SARIMA_ORDER,
    seasonal_order=SEASONAL_ORDER
)
```

</details>

#### Plot SARIMA-Prognose mit Konfidenzintervall

In [ ]:
# Plot
plt.figure(figsize=(14, 5))
plt.plot(y_train.index[-plot_days:], y_train[-plot_days:], label="Training", color="blue")
plt.plot(y_test.index, y_test, label="Tatsächliche Werte", color="black")
plt.plot(arima_forecast.index, arima_forecast, label="ARIMA Forecast", color="green")
plt.plot(sarima_forecast.index, sarima_forecast, label="SARIMA Forecast", color="red", linewidth=2)
plt.fill_between(sarima_forecast.index, sarima_conf_int.iloc[:, 0], sarima_conf_int.iloc[:, 1], color="red", alpha=0.2, label="95%-Konfidenzintervall")
plt.title("SARIMA Forecast mit 95%-Konfidenzintervall")
plt.xlabel("Zeit")
plt.ylabel("Strombedarf")
plt.legend()
plt.grid(True, alpha=0.8)
plt.tight_layout()
plt.show()

Aufgaben zur Reflexion

1. Wo macht ARIMA systematische Fehler?
2. Welche Muster erkennt das Modell offenbar nicht?
3. Welche Erweiterung könnte helfen?

### 2.3 SARIMAX trainieren: SARIMA + exogene Variablen

In [ ]:
# Aufgabe:
# Wählt exogene Variablen anhand von Fachwissen aus und begründet die Wahl.
# - produktion_kwh:
# - is_weekend:
# - sin_week:
# - cos_week:
# - weitere?

# Hinweis zu Kalenderfeaturen
# Diese Variablen geben dem SARIMAX zusätzliche Kalenderinformationen. 
# is_weekend sagt dem Modell, ob Wochenende ist. 
# sin_week und cos_week beschreiben den Wochentag zyklisch, sodass Montag und Sonntag als benachbarte Tage erkannt werden. 
# sin_year und cos_year machen dasselbe für den Jahresverlauf und helfen, saisonale Effekte wie Sommer und Winter abzubilden
idx = model_df.index
model_df["is_weekend"] = (idx.dayofweek >= 5).astype(int)     
model_df["sin_week"] = np.sin(2 * np.pi * idx.dayofweek / 7)     
model_df["cos_week"] = np.cos(2 * np.pi * idx.dayofweek / 7)
model_df["sin_year"] = np.sin(2 * np.pi * idx.dayofyear / 365.25) 
model_df["cos_year"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)
# erneut Trainings-/Test-Split anwenden
train = model_df.iloc[:-TEST_DAYS]
test = model_df.iloc[-TEST_DAYS:]

exog_cols = [_, _, ...]

x_train = train[exog_cols]
x_test = test[exog_cols]
# Skalierung nur auf Trainingsdaten fitten, um Data Leakage zu vermeiden
scaler = StandardScaler()
x_train_scaled = pd.DataFrame(scaler.fit_transform(x_train), index=x_train.index, columns=x_train.columns)
x_test_scaled = pd.DataFrame(scaler.transform(x_test), index=x_test.index, columns=x_test.columns)

sarimax_model = SARIMAX(
    y_train,
    exog=x_train_scaled,
    order=SARIMA_ORDER,
    seasonal_order=SEASONAL_ORDER
)
sarimax_result = sarimax_model.fit(disp=False)

# Forecast
# Hinweis:
# Für den SARIMAX-Forecast müssen zukünftige Werte der exogenen Variablen vorliegen.
# In dieser HandsOn Übung verwenden wir die tatsächlich beobachteten exogenen Werte aus dem Testzeitraum.
# Das ist ein sogenannter Conditional Forecast.
# In einer produktiven Anwendung müssten diese Werte z.B. durch Wetterprognosen,
# Erzeugungsprognosen oder separate Forecast-Modelle bereitgestellt werden.
sarimax_forecast_result = sarimax_result.get_forecast(
    steps=len(y_test),
    exog=x_test_scaled
)

# Forecast erzeugen
sarimax_forecast = sarimax_forecast_result.predicted_mean
sarimax_conf_int = sarimax_forecast_result.conf_int(alpha=0.05)
sarimax_forecast.index = y_test.index
sarimax_conf_int.index = y_test.index
forecasts["SARIMAX"] = sarimax_forecast

print("SARIMAX trainiert.")
display(sarimax_result.params.loc[[p for p in sarimax_result.params.index if p in exog_cols]].to_frame("Koeffizient"))

<details>
  <summary>Lösung anzeigen</summary>

```python
exog_cols = ["produktion_kwh", "is_weekend", "sin_week", "cos_week"]
```

</details>

#### Interpretation der exogenen Variablen

In [ ]:
# Interpretation der exogenen Variablen
exog_params = pd.DataFrame({
    "Koeffizient": sarimax_result.params,
    "p-Wert": sarimax_result.pvalues
})
exog_params = exog_params.loc[exog_params.index.intersection(exog_cols)]
exog_params["Signifikant auf 5%-Niveau"] = exog_params["p-Wert"] < 0.05
display(exog_params)

#### Plot SARIMAX-Prognose mit Konfidenzintervall

In [ ]:
# Plot
plt.figure(figsize=(14, 5))
plt.plot(y_train.index[-plot_days:], y_train[-plot_days:], label="Training", color="blue")
plt.plot(y_test.index, y_test, label="Tatsächliche Werte", color="black")
plt.plot(sarima_forecast.index, sarima_forecast, label="SARIMA Forecast", color="green", linewidth=2)
plt.plot(sarimax_forecast.index, sarimax_forecast, label="SARIMAX Forecast", color="red", linewidth=2)
plt.fill_between(sarimax_forecast.index, sarimax_conf_int.iloc[:, 0], sarimax_conf_int.iloc[:, 1], color="red", alpha=0.2, label="95%-Konfidenzintervall")
plt.title("SARIMAX Forecast mit 95%-Konfidenzintervall")
plt.xlabel("Zeit")
plt.ylabel("Strombedarf")
plt.legend()
plt.grid(True, alpha=0.8)
plt.tight_layout()
plt.show()

Aufgaben zur Reflexion

1. Hat sich der Forecast gegenüber SARIMA verbessert?
2. Welche exogenen Variablen scheinen hilfreich und welche eher nicht?
2. In welchen Business-Szenarien ist SARIMAX sinnvoller als SARIMA?
3. Was kann gegen den Einsatz von SARIMAX sprechen, auch wenn es ggf. einen besseren Forecast liefert?

## 3. Vergleich der Modelle

In [ ]:
# Vergleichslogik:
# - ARIMA als Referenzmodell
# - SARIMA vs. ARIMA: Bringt explizite Saisonalität Mehrwert?
# - SARIMAX vs. SARIMA: Bringen exogene Variablen zusätzlichen Mehrwert?

# Fehlermaße berechnen
rmse_arima = np.sqrt(mean_squared_error(y_test, arima_forecast))
rmse_sarima = np.sqrt(mean_squared_error(y_test, sarima_forecast))
rmse_sarimax = np.sqrt(mean_squared_error(y_test, sarimax_forecast))
results = pd.DataFrame({
    "Modell": ["ARIMA", "SARIMA", "SARIMAX"],
    "RMSE": [rmse_arima, rmse_sarima, rmse_sarimax]
})
print(results)

# Verbesserungen in Prozent berechnen
improvement_sarima = (rmse_arima - rmse_sarima) / rmse_arima * 100
improvement_sarimax = (rmse_sarima - rmse_sarimax) / rmse_sarima * 100
print(f"\nRMSE Verbesserung durch Erweiterung um Saisonalität: {improvement_sarima:.2f}%")
print(f"\nRMSE Verbesserung durch Erweiterung um exogene Variablen: {improvement_sarimax:.2f}%")

In [ ]:
# Visualisierung der Forecasts
ax = forecasts[["Ist", "ARIMA", "SARIMA", "SARIMAX"]].plot(figsize=(18, 6),linewidth=1.3)
ax.set_title("Forecast-Vergleich im Testzeitraum")
ax.set_ylabel("Hausverbrauch [kWh/Tag]")
ax.grid(True, alpha=0.8)
plt.tight_layout()
plt.show()

Aufgabe: Was fällt beim Vergleich der Modelle & deren Forecasts auf?

# 11. Kritische Interpretation für die Abgabe / Diskussion

Aufgabe: Das Ergebnis soll dem Geschäftsführer präsentiert werden. Beantwortet dazu folgende Fragen

1. Welches Modell würden Sie empfehlen?
2. Warum?
3. Welche Risiken bestehen?
4. Welche zusätzlichen Daten würden Sie künftig erfassen?


Aufgabe: Bewerte die Modelle mittels einer Bewertungsmatrix

| Kriterium | ARIMA | SARIMA | SARIMAX |
|------------|------------|------------|------------|
| RMSE | | | |
| Interpretierbarkeit | | | |
| Datenbedarf | | | |
| Wartungsaufwand | | | |
| Benötigt zukünftige Exogene | | | |
| Business-Empfehlung | | | |

Aufgabe: Beantwortung der Forschungsfrage

> **Forschungsfrage: Verbessern Saisonalität & exogene Informationen den Forecast des täglichen Hausverbrauchs gegenüber einem ARIMA-Modell?**